# Prompt evaluation

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## ATENTION

- Prefer "claude-4-5-haiku" for this notebook.

## 0. Setup

### 0.1. Env and Client

In [1]:
# Load env variables and create client

## 1 Env
from dotenv import load_dotenv

load_dotenv()

## 2 Client
from anthropic import Anthropic

client = Anthropic()
model = "bedrock/anthropic.claude-4-5-haiku"

### 0.2. Helper functions

In [2]:
# Helper functions
def add_user_message(
    messages, 
    text,
):
    user_message = {
        "role": "user", 
        "content": text,
    }
    
    messages.append(user_message)


def add_assistant_message(
    messages, 
    text,
):
    assistant_message = {
        "role": "assistant", 
        "content": text,
    }

    messages.append(assistant_message)


def chat(
    messages, 
    system=None, 
    temperature=1.0, 
    stop_sequences=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

## Prompt Eval

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

In [4]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket'). The function should return the region if present, or 'us-east-1' as the default."}, {'task': 'Create a JSON CloudFormation template snippet that defines an IAM role with a trust policy allowing the EC2 service to assume it.'}, {'task': "Write a regular expression that validates an AWS access key ID format (starts with 'AKIA' followed by 20 alphanumeric characters)."}]


In [5]:
with open('dataset.json', 'w') as f:
    json.dump(
        dataset, 
        f, 
        indent=2,
    )

print("Dataset saved to dataset.json")

Dataset saved to dataset.json
